In [11]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain.agents.middleware import wrap_model_call
import subprocess
from pathlib import Path
import sys
from typing import Optional

load_dotenv()

True

In [12]:
embeddings= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2611.06it/s]


In [13]:
django_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="django_docs",
    embedding_function=embeddings
)

python_vectorstore= Chroma(
    persist_directory="./django_chroma_db",
    collection_name="python_scripts",
    embedding_function=embeddings
)

In [14]:
django_db= django_vectorstore.get()
python_db=python_vectorstore.get()

django_splits=[
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(django_db["documents"], django_db["metadatas"])
]

python_splits = [
    Document(page_content=text, metadata=meta or {})
    for text, meta in zip(python_db["documents"], python_db["metadatas"])
]

django_retriever= django_vectorstore.as_retriever(search_kwargs={"k":4})
python_retriever= python_vectorstore.as_retriever(search_kwargs={"k":4})


all_splits= django_splits + python_splits
print(f"Loaded {len(all_splits)} total splits ({len(django_splits)} Django docs + {len(python_splits)} Python codebase).")
bm25_retriever= BM25Retriever.from_documents(all_splits)


Loaded 6361 total splits (5110 Django docs + 1251 Python codebase).


In [15]:
bm25_retriever.k=8

In [16]:
#Creating hybrid retriever
hybrid_retriever= EnsembleRetriever(
    retrievers=[django_retriever, python_retriever, bm25_retriever],
    weights=[0.4,0.3,0.3]#40% django sementic search, 30% python sementic and keyword
    
)

In [17]:
#Reranker block of code
reranker= CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2639.95it/s]


## Generation Tools

In [18]:
DEFAULT_DIR = r"C:\Users\ZENOID\Desktop\Home\home\self_made.projects\AI(created by me) generated projects"

@tool
def retrieve_django_content(query: str) -> str:
    """
    Search the Django knowledge base for documentation, code examples,
    debugging information, and practical examples.
    Use this tool to gain more context and information needed to answer the users question
    """
    print("Using retriever......")
    docs = hybrid_retriever.invoke(query)

    print("Reranking......")
    pairs = [[query, doc.page_content] for doc in docs]
    scores = reranker.predict(pairs)
    scored_docs = sorted(
        zip(docs, scores),
        key=lambda x: x[1],
        reverse=True
    )
    top_docs = scored_docs[:3]

    return "\n\n".join(doc.page_content for doc, score in top_docs)


def _get_bin_dir(env_path: Path) -> Path:
    """Returns the folder inside a venv where executables live (differs by OS)."""
    return env_path / "Scripts" if sys.platform == "win32" else env_path / "bin"


def _exe(bin_dir: Path, name: str) -> Path:
    """Adds .exe to the executable name only on Windows."""
    return bin_dir / (f"{name}.exe" if sys.platform == "win32" else name)


@tool
def setup_django_project(project_name: str,app_name: Optional[list[str] | str] = None,directory: Optional[str] = None) -> dict:
    """
    Set up a Django project safely.
    This operation is idempotent: existing virtual environments,
    Django projects, and apps are detected and will not be recreated.
    """
    if not project_name or not project_name.strip():
        return {"status": "error","message": "project_name is required"}

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    target_path = Path(directory) / project_name
    target_path.mkdir(parents=True, exist_ok=True)

    results = {
        "status": "success",
        "project_name": project_name,
        "project_path": str(target_path),
        "virtualenv": None,
        "django": None,
        "project": None,
        "apps": []
    }

    # Creating Virtual environment
    env_path = target_path / "env"
    if env_path.exists():
        results["virtualenv"] = "already_exists"
    else:
        print("Creating virtual environent......")
        result = subprocess.run(
            [sys.executable, "-m", "venv", "env"],cwd=target_path,capture_output=True,text=True
        )
        if result.returncode != 0:
            return {
                "status": "error",
                "step": "create_virtualenv",
                "message": result.stderr[-1000:]
            }
        results["virtualenv"] = "created"

    # Getting executables
    bin_dir = _get_bin_dir(env_path)
    python_exe = _exe(bin_dir, "python")
    pip_exe = _exe(bin_dir, "pip")

    # Checking if Django is already installed
    django_check = subprocess.run(
        [str(python_exe), "-c", "import django"],capture_output=True,text=True
    )
    if django_check.returncode == 0:
        results["django"] = "already_installed"
    else:
        print("Installing Django......")
        install_result = subprocess.run(
            [str(pip_exe), "install", "django"],cwd=target_path,capture_output=True,text=True
        )
        if install_result.returncode != 0:
            return {
                "status": "error",
                "step": "install_django",
                "message": install_result.stderr[-1000:]
            }
        results["django"] = "installed"


    # Creating Django Project
    manage_py = target_path / "manage.py"
    project_package = target_path / project_name
    if manage_py.exists() and project_package.exists():
        results["project"] = "already_exists"
    else:
        print("Creating Django project......")
        django_admin = _exe(bin_dir, "django-admin")
        project_result = subprocess.run(
            [str(django_admin),"startproject",project_name,"."],cwd=target_path,capture_output=True,text=True
        )
        if project_result.returncode != 0:
            return {
                "status": "error",
                "step": "create_project",
                "message": project_result.stderr[-1000:]
            }
        results["project"] = "created"


    # Creating Django Apps
    if app_name:
        print("Creating Django Apps......")
        apps = (app_name if isinstance(app_name, list) else [app_name])
        for app in apps:
            app_path = target_path / app
            if app_path.exists():
                results["apps"].append({
                    "name": app,
                    "status": "already_exists"
                })
                continue
            app_result = subprocess.run(
                [str(python_exe),"manage.py","startapp",app],cwd=target_path,capture_output=True,text=True
            )
            if app_result.returncode != 0:
                results["apps"].append({
                    "name": app,
                    "status": "failed",
                    "error": app_result.stderr[-500:]
                })
                results["status"] = "partial_success"
            else:
                results["apps"].append({
                    "name": app,
                    "status": "created"
                })
    return results

_active_server_process = None


@tool
def manage_server(action: str,project_name: Optional[str] = None,directory: Optional[str] = None) -> dict:
    """
    Starts or stops the active Django development server.
    action must be either "start" or "stop".
    For "start", project_name identifies the Django project.
    For "stop", no project_name is required.
    """
    global _active_server_process
    action = action.strip().lower()
    if action not in ("start", "stop"):
        return {
            "status": "error",
            "action": action,
            "message": "Invalid action. Use 'start' or 'stop'."
        }

    if not directory or directory.strip() in ("", "."):
        directory = DEFAULT_DIR

    results = {
        "action": action,
        "project_name": project_name,
        "project_path": None,
        "status": "success"
    }
    # Stops Django Server
    if action == "stop":
        if _active_server_process and _active_server_process.poll() is None:
            _active_server_process.terminate()
            try:
                _active_server_process.wait(timeout=3)
            except subprocess.TimeoutExpired:
                _active_server_process.kill()
            _active_server_process = None
            results["message"] = (
                "Django development server stopped successfully."
            )
        else:
            results["status"] = "not_running"
            results["message"] = (
                "No active Django server is currently running."
            )
        return results


    if not project_name:
        return {
            "status": "error",
            "action": "start",
            "message": "project_name is required when starting the server."
        }

    target_path = Path(directory) / project_name

    # Fallback in case directory itself contains manage.py
    if not (target_path / "manage.py").exists():

        if (Path(directory) / "manage.py").exists():
            target_path = Path(directory)

        else:
            return {
                "status": "error",
                "action": "start",
                "project_name": project_name,
                "message": (
                    f"Could not find manage.py for "
                    f"project '{project_name}'."
                )
            }

    manage_py = target_path / "manage.py"

    results["project_path"] = str(target_path)
    env_path = target_path / "env"
    python_exe = _exe(_get_bin_dir(env_path),"python")

    if not python_exe.exists():

        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "Could not find the virtual environment Python executable."
            )
        }

    #Checking if Server is running
    if _active_server_process and _active_server_process.poll() is None:
        return {
            "status": "already_running",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": (
                "A Django development server is already running."
            )
        }

    #Start Django Server

    try:
        _active_server_process = subprocess.Popen([str(python_exe),"manage.py","runserver","--noreload"],cwd=target_path,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
        results["status"] = "started"
        results["message"] = f"Django server started successfully for project '{project_name}'."
        return results
    except Exception as error:
        return {
            "status": "error",
            "action": "start",
            "project_name": project_name,
            "project_path": str(target_path),
            "message": str(error)
        }

## File Editor Tools
Prompt/Task -> Search -> Create/Read -> Edit -> Write

In [19]:
@tool
def search_target(query: str, directory: str) -> str:
    """Searches through existing files in current directory to extract information from codebase based on the query/Users question, and returns all the file names/directory that contain the query/Users question"""
    path = Path(directory)
    print(f"Searching for {query} in {directory}.......")
    if not path.exists():
        return f"Directory {directory} does not exist"

    files = path.glob("**/*.py")
    target_files = []
    for file in files:
        # Skip anything inside a virtual environment folder
        if "env" in file.parts or "venv" in file.parts or "site-packages" in file.parts:
            continue

        with open(file, "r", encoding="utf-8") as f:
            content = f.read()
            if query in content:
                target_files.append(str(file))
                print(f"Found {query} in {file}")

    if len(target_files) == 0:
        return f"Could not find {query} in any files"
    return "\n".join(target_files)


@tool
def deeper_search(query: str, directory: Optional[str] = None) -> str:
    """Searches deeper through existing folders and files inside the folders in current directory to extract information from codebase based on the query/Users question, and returns all the file names/directory that contain the query/Users question"""
    print("Enabling deeper search......")
    if not directory or directory.strip() in (".", ""):
        directory = DEFAULT_DIR
    path = Path(directory)
    print(f"Searching for {query} in {directory}.......")
    folders=path.glob("**/")
    target_folders=[]
    for folder in folders:
        if folder.is_dir():
            if query in folder.name:
                target_folders.append(str(folder))
                print(f"Found {query} in {folder}")
    if len(target_folders) == 0:
        return f"Could not find {query} in any folders"
    return "\n".join(target_folders)


@tool
def read_file(file_path: str) -> str:
    """Reads and return the full content of a file so it can be reviewed or editted"""
    path = Path(file_path)
    print(f"Reading File at {file_path}.......")
    if not path.exists():
        return f"File not found: {file_path}"
    return path.read_text(encoding="utf-8")

@tool
def edit_file(file_path: str, old_code: str, new_code: str) -> str:
    """
    Makes a targeted edit to an existing file by replacing an exact block of
    existing code (old_code) with new code (new_code). old_code must match
    the file's content exactly, including whitespace, and must appear only once.
    """
    path = Path(file_path)
    print(f"Editing File at {file_path}.......")
    if not path.exists():
        return f"File not found: {file_path}"

    content = path.read_text(encoding="utf-8")

    count = content.count(old_code)
    if count == 0:
        return "old_code was not found in the file — no changes made."
    if count > 1:
        return f"old_code appears {count} times in the file — it must be unique. No changes made."

    updated_content = content.replace(old_code, new_code)
    path.write_text(updated_content, encoding="utf-8")
    return f"Successfully edited {file_path}"

@tool
def write_new_file(file_path: str, content: str, overwrite: bool = False) -> str:
    """
    Creates a new file with the given content. If a file already exists at
    that path, it will NOT be overwritten unless overwrite=True is explicitly set.
    Use this for creating brand new files that don't already exist
    (e.g. serializers.py, a new utility script) — not for modifying files
    Django already generated, use append_to_file or edit_file for those.
    """
    path = Path(file_path)
    print(f"Writing New File at {file_path}.......")
    if path.exists() and not overwrite:
        return f"File already exists at {file_path}. Use overwrite=True if you really want to replace it, or use edit_file/append_to_file to modify it instead."

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
    return f"Wrote new file: {file_path}"


@tool
def append_to_file(file_path: str, code: str) -> str:
    """
    Adds new code to the END of an existing file, without touching what's
    already there. Use this to add a new view, function, or class to a file
    that already has content — e.g. adding a new function to views.py.
    """
    path = Path(file_path)
    print(f"Appending to File at {file_path}.......")
    if not path.exists():
        return f"File not found: {file_path}. Use write_file to create it first."

    with open(path, "a", encoding="utf-8") as f:
        f.write("\n\n" + code)
    return f"Appended new code to {file_path}"

# @tool
# def delete_file():
#     pass

In [20]:
#Constructing Agent and memory
llm=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)
memory=MemorySaver()



In [21]:
#Deleting memory history/cache
# memory.storage.pop("test-8", None)

In [22]:
#Trimming users messages to preserve context window
@wrap_model_call
def limit_history(request, handler):
    messages = request.state["messages"]

    recent_messages = messages[-2:]

    request.state["messages"] = recent_messages

    return handler(request)

In [23]:
agent= create_agent(
    model=llm,
    tools=[retrieve_django_content, setup_django_project, manage_server, search_target,read_file,edit_file,append_to_file,write_new_file, deeper_search],
    system_prompt = r"""
You are a Django engineering assistant with two jobs: reviewing Django code, and setting up/running Django projects using the tools available to you.

You have two categories of tools:

1. KNOWLEDGE TOOL — retrieve_django_content
   Call this when the user asks a question, wants a code review, or wants an explanation of a Django concept, pattern, or error — anything where getting it right depends on accurate technical detail.
   Do NOT call this for simple action requests that don't require technical judgment — e.g. "create a project called X," "create an app called Y," "start the server," "stop the server." These just need the right tool called correctly, not a documentation lookup.
   If a request mixes both (e.g. "review this code, then create a project"), call retrieve_django_content only for the review part.

2. ACTION TOOLS — these perform real actions on the user's machine:
   - setup_project(project_name, apps, directory): Bootstraps a complete Django setup in one step (creates virtualenv, installs Django, initializes project, and generates specified apps).
   - manage_server(action, project_name, directory): Controls the Django development server by specifying an action such as "start" or "stop".

   DIRECTORY RULE:
   - If the user's request doesn't specify a directory, leave the `directory` parameter out of the tool call entirely — do not fill it with "." or the current workspace path. The tool itself will fall back to the correct default folder.
   - Never guess or invent a directory path on your own when none is given.

STRICT ORDER OF OPERATIONS (for action requests):
1. create_virtualenv → 2. create_django_project → 3. create_django_app (only if an app was requested) → 4. run_django_server
   Skip a step only if the user says it already exists.

RULES FOR ACTION TOOLS:
- If the user asks you to create, set up, scaffold, start, or run a Django project or app — use the tools. Do not just describe the steps in text; call the tools.
- Infer sensible defaults if the user is vague (e.g. project name "myproject") and state the assumption in one short sentence before acting.
- After calling a tool, report what actually happened in one or two sentences — summarize, don't paste the tool's raw return string verbatim.
- If a tool call fails or the server doesn't start, say so plainly and suggest the likely cause — do not pretend it succeeded.
- Never call run_django_server without first confirming a project actually exists at that path.
- Never call create_django_app without first confirming the target project actually exists.
- Only call stop_django_server if the user asks to stop, restart, or if you are about to start a new server and one may already be running.
- If the user only asks you to explain what steps you'd take, describe them in words — do not call any tool.
- Call retrieve_django_content for: questions, code reviews, explanations of Django concepts, AND any time you're diagnosing a bug whose cause isn't immediately obvious from the code alone (e.g. behavior that only appears under certain conditions, subtle state/mutation issues, ORM query behavior). Debugging is not a pure action — ground your diagnosis before proposing a fix.

FILE TOOLS — for reading and modifying code that already exists on disk:

- search_target(query, directory): use this FIRST when the user refers to existing code but doesn't give you an exact file path (e.g. "fix the bug in my products view," "where is ALLOWED_HOSTS set"). It searches all .py files under a directory for a text match and returns which files contain it. Use its result to get the real file path before calling read_file.
- deeper_search(query, directory): Use this when the user refers to existing code but doesn't give you an exact file path (e.g. "fix the bug in my products view," "where is ALLOWED_HOSTS set") or the search_target tool doesn't return any results. It searches all folders under a directory and returns which folders contain the query. Use its result to get the real file path before calling read_file.
- read_file(file_path): ALWAYS call this before editing or appending to any file. Never guess file content from memory or from earlier in the conversation — content may have changed.
- edit_file(file_path, old_code, new_code): use to change something that already exists (a specific line, a setting, a function body). old_code must be copied EXACTLY from what read_file returned, including whitespace and indentation, and must be unique in the file. If edit_file reports "not found" or "appears more than once," call read_file again and adjust old_code — do not guess repeatedly.
- append_to_file(file_path, code): use to add brand-new code (a new function, view, or class) to the END of a file that already has content, without disturbing what's there. Do NOT use this for files with a structure that must stay intact, like urls.py's urlpatterns list or settings.py — use edit_file for those so the new code lands in the right place, not just tacked onto the end.
- write_new_file(file_path, content, overwrite=False): use only to create a file that does not exist yet (e.g. a new serializers.py or utils.py). If it reports the file already exists, switch to edit_file or append_to_file instead — do not retry with overwrite=True unless the user explicitly asked to replace the entire file.

WORKFLOW FOR CODE CHANGES:
1. If you don't have the exact file path, call search_target to find it.
2. Call read_file to see the current content.
3. Decide: is this a new file (write_new_file), new code added to an existing file (append_to_file), or a change to something that already exists (edit_file)?
4. Make the change, then report in plain words what was changed — do not paste the tool's raw return string verbatim.
5. If a step fails (file not found, old_code not unique, etc.), don't guess or retry blindly — re-read the file and adjust, or tell the user what went wrong.

RULES FOR CODE REVIEW / KNOWLEDGE ANSWERS:
- Answer using the context retrieved from retrieve_django_content. If it doesn't return anything relevant, say "I don't have this information in my knowledge base" rather than guessing.
- Keep review answers direct: lead with the answer, then 1-3 sentences of essential context.
- Use bullet points only when the user asks for a list of issues/items.
- Don't narrate your retrieval or tool-selection process to the user (no "I searched the knowledge base..." or "I'm now calling retrieve_django_content...") — just do it and report the outcome.
- retrieve_django_content only covers Django documentation and technical content — never call it for questions about this conversation's own history or what was previously created; answer those from the conversation itself, or say plainly if you don't have that information.

GENERAL:
- Be concise. No filler like "Based on the above" or "In conclusion."
- If a request is ambiguous between "explain how to do X" and "do X for me," default to doing it if action tools are relevant — ask only if the ambiguity would cause you to act on the wrong project/directory.
""",
    middleware=[limit_history],
    checkpointer=memory
)

#Memory
config = {"configurable": {"thread_id": "test-23"}}

In [25]:
#Creatin User Interface
while True:
    question= input("Any Question about django: ").strip()
    if question.lower() in ["exit", "quit", "q"]:
        print("See you soon...")
        break
    if not question:
        continue

    result= agent.invoke({
        "messages": [HumanMessage(content=question)]
    }, config=config)

    print("\n Final Result")
    print(result["messages"][-1].content)


 Final Result
The Django development server has been stopped successfully. Let me know if you need any further assistance.
See you soon...
